<a href="https://colab.research.google.com/github/Bebarzzz/machine-/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# The models
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('Chronotype_NHANES_Imputation1.csv')
df.head()



,Seqn,Gender,Age,Race,BMI,Waist_C,Systolic,Diastolic,Carb_diet,HSCRP,Smokingstatus,Alcohol,Sleep_hrs,Sleeptime,Wakeuptime,Chronotype_slphrs,WakeUpCat
0,83732,1,62,2.0,27.8,101.1,122.6667,65.33334,126.0,0.6,3.0,1.0,5.5,23:30:00,05:00:00,3.0,1
1,83733,1,53,2.0,30.8,107.9,140.0000,86.00000,126.0,1.4,1.0,7.0,8.0,23:00:00,07:00:00,3.0,3
2,83734,1,78,2.0,28.8,116.5,135.3333,45.33333,96.0,0.6,3.0,0.0,7.0,22:30:00,05:30:00,2.0,2
3,83735,2,56,2.0,42.4,110.1,134.0000,70.00000,216.0,9.0,3.0,3.0,6.5,23:30:00,06:00:00,3.0,2
4,83741,1,22,3.0,28.0,86.6,111.3333,72.66666,5.5,1.3,2.0,3.0,6.5,23:00:00,05:30:00,3.0,2


### Data Preparation
We need to separate the target variable `Chronotype_slphrs` from the features. We should also remove identifiers like `Seqn` and non-numeric time columns that aren't ready for modeling.

In [ ]:
# Define features and target
# Dropping Seqn (ID) and the original time strings which aren't numeric
X = df.drop(columns=['Chronotype_slphrs', 'Seqn', 'Sleeptime', 'Wakeuptime'])
y = df['Chronotype_slphrs']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (4469, 13)
Testing set shape: (1118, 13)


### Model Training and Evaluation
Now we initialize and fit the Random Forest Classifier.

In [ ]:
# Initialize the revised model with balanced class weights and tuned depth
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)

# Fit the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluation
print("Revised Random Forest Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Revised Random Forest Accuracy Score: 0.740608228980322

Classification Report:
              precision    recall  f1-score   support

         1.0       0.71      0.51      0.59       156
         2.0       0.70      0.76      0.73       344
         3.0       0.78      0.86      0.82       443
         4.0       0.67      0.62      0.64        74
         5.0       0.80      0.58      0.67       101

    accuracy                           0.74      1118
   macro avg       0.73      0.67      0.69      1118
weighted avg       0.74      0.74      0.74      1118



In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)


### XGBoost Classifier
Next, we'll try the XGBoost model. First, we need to import it and then train it using our prepared features and target.

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost often requires label encoding for targets starting from 0
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Initialize the XGBoost model (removed deprecated use_label_encoder)
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')

# Fit the model
xgb_model.fit(X_train, y_train_encoded)

# Make predictions
y_pred_xgb = xgb_model.predict(X_test)

# Evaluation
print("XGBoost Accuracy Score:", accuracy_score(y_test_encoded, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test_encoded, y_pred_xgb, target_names=le.classes_.astype(str)))

XGBoost Accuracy Score: 0.7397137745974955

Classification Report:
              precision    recall  f1-score   support

         1.0       0.64      0.49      0.56       156
         2.0       0.70      0.70      0.70       344
         3.0       0.78      0.88      0.83       443
         4.0       0.77      0.68      0.72        74
         5.0       0.78      0.67      0.72       101

    accuracy                           0.74      1118
   macro avg       0.73      0.69      0.71      1118
weighted avg       0.74      0.74      0.73      1118



In [ ]:
byy